In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random
import os

In [3]:
def extract_petri_dish(image):
    #convert the picture to gray if it wasn't already
    gray=cv2.cvtColor(image,cv2.COLOR_BGR2GRAY)
    #reducing the noise
    blurred=cv2.GaussianBlur(gray,(11,11),0)
    # Use adaptive thresholding to handle different lighting conditions
    # _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    thresh = cv2.adaptiveThreshold(blurred, 255,
                                   cv2.ADAPTIVE_THRESH_MEAN_C,
                                   cv2.THRESH_BINARY_INV, 51, 5)
    kernel = np.ones((5, 5), np.uint8)
    closed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    contours = sorted(contours, key=cv2.contourArea, reverse=True)

    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if 0.8 < w / h < 1.2 and w > 200:
            size = max(w, h)
            cx, cy = x + w // 2, y + h // 2
            half = size // 2

            x1 = max(0, cx - half)
            y1 = max(0, cy - half)
            x2 = min(image.shape[1], cx + half)
            y2 = min(image.shape[0], cy + half)

            cropped = image[y1:y2, x1:x2]
            return cropped

    h, w = image.shape[:2]
    side = min(h, w)
    startx = (w - side) // 2
    starty = (h - side) // 2
    return image[starty:starty + side, startx:startx + side]

In [4]:
# Define input and output folders
input_folder = r"E:\Github\Block A 2\Block B\Task 1\Week 2"
output_folder = os.path.join(input_folder, "petri_dish")

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# List of image files only (filtering common image extensions)
image_files = [f for f in os.listdir(input_folder) 
               if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]

# Randomly select N files to process
N = 5  # Change this to however many images you want to process
selected_files = random.sample(image_files, min(N, len(image_files)))

# Process and save each selected file
for filename in selected_files:
    img_path = os.path.join(input_folder, filename)
    img = cv2.imread(img_path)

    if img is None:
        print(f"Failed to load image: {filename}")
        continue

    petri_dish = extract_petri_dish(img)
    
    save_path = os.path.join(output_folder, filename)
    cv2.imwrite(save_path, petri_dish)
    print(f"Processed and saved: {filename}")

Processed and saved: train_Dean_242715_im1.png
Processed and saved: train_Dean_242467_im1.png
Processed and saved: train_Alican_231131_im4.png
Processed and saved: train_Dean_240046_im1.png
Processed and saved: train_Dean_221350_im1.png


In [8]:
for filename in image_files:
    img_path = os.path.join(output_folder, filename)
    img = cv2.imread(img_path)

    if img is not None:
        print(f"{filename}: {img.shape[:2]}")


train_Alican_231131_im4.png: (2824, 2824)
train_Dean_221350_im1.png: (2842, 2842)
train_Dean_240046_im1.png: (2822, 2822)
train_Dean_242467_im1.png: (2842, 2842)
train_Dean_242715_im1.png: (2824, 2824)
